In [2]:
from langchain_community.vectorstores import FAISS
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader

C:\Users\mrraj\AppData\Local\Temp\ipykernel_6256\1723707667.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


KeyboardInterrupt: 

In [ ]:
loader = TextLoader("rag_knowledge.txt")
docs = loader.load()
docs

[Document(metadata={'source': 'rag_knowledge.txt'}, page_content='Retrieval Augmented Generation is commonly known as RAG.\nRAG combines information retrieval with large language models.\nA RAG system retrieves relevant information before generating an answer.\nThe retrieved information is provided to the language model as context.\nRAG is useful when a language model needs access to external knowledge.\nExternal knowledge can include documents, websites, databases, manuals, reports, and company information.\nRAG can reduce the need to retrain a language model whenever new information becomes available.\nA typical RAG pipeline contains document loading, text splitting, embedding generation, vector storage, retrieval, and generation.\nSome advanced RAG systems also include query enhancement, reranking, hybrid search, filtering, and evaluation.\nThe quality of retrieval has a significant impact on the quality of the final answer.\nIf the retriever returns irrelevant documents, the langua

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50
)
chunks = splitter.split_documents(docs)
chunks

In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)
vectorestore = FAISS.from_documents(chunks,embedding)

retriever = vectorestore.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":5}
)

retriever

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E8C22F78C0>, search_type='mmr', search_kwargs={'k': 5})

In [ ]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(
    model="groq:openai/gpt-oss-120b"
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E469B57620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E46B728050>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser


query_prompt = ChatPromptTemplate.from_template(
    """
    You are a query enhancement assistant.

    Rewrite the user's question into a clear and detailed
    search query for a RAG system.

    Rules:
    - Preserve the original meaning.
    - Add useful context when necessary.
    - Include important technical terms.
    - Remove unnecessary conversational words.
    - Do not answer the question.
    - Return only the enhanced query.

    User Query:
    {query}
    """
)

query_expansion_chain = query_prompt | llm | StrOutputParser()
query_expansion_chain

ChatPromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template="\n    You are a query enhancement assistant.\n\n    Rewrite the user's question into a clear and detailed\n    search query for a RAG system.\n\n    Rules:\n    - Preserve the original meaning.\n    - Add useful context when necessary.\n    - Include important technical terms.\n    - Remove unnecessary conversational words.\n    - Do not answer the question.\n    - Return only the enhanced query.\n\n    User Query:\n    {query}\n    "), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs':

In [ ]:
query_expansion_chain.invoke({"query":"Langchain"})

'LangChain library overview, core features, and usage examples for building LLM-powered applications in Python, including chain composition, agents, memory management, and integration with language models and vector stores.'

In [ ]:
answer_prompt = PromptTemplate.from_template(
    """
    Answer the question Based on Context Below.
    
    Context:
    {context}
    
    
    Question:{input} 
    """
)

document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=answer_prompt
)

In [ ]:
from langchain_core.runnables import RunnableMap

rag_pipeline = RunnableMap({
    "input": lambda x: x["input"],

    "context": lambda x: retriever.invoke(
        query_expansion_chain.invoke({
            "query": x["input"]
        })
    )
}) | document_chain

In [ ]:
query = {"input":"what type of memory does langchain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print(response)

Supported memory types and implementations in the LangChain library (e.g., ConversationBufferMemory, ConversationSummaryMemory, VectorStoreMemory, RedisCache, InMemory, SQL, and other persistent or transient memory modules)
LangChain includes a family of **memory** components that let a chain keep track of what has been said (or retrieved) across turns.  
The main built‑in memory types are:

| Memory class | What it does |
|--------------|--------------|
| **ConversationBufferMemory** | Stores the full chat history as a plain text buffer. |
| **ConversationBufferWindowMemory** | Keeps only the most recent *k* turns (a sliding window). |
| **ConversationSummaryMemory** | Summarizes the conversation on‑the‑fly so the LLM sees a concise recap instead of the full transcript. |
| **ConversationSummaryBufferMemory** | Combines a rolling buffer with periodic summarisation. |
| **ConversationTokenBufferMemory** | Limits stored history by token count rather than by turn count. |
| **Conversatio

In [ ]:
query = {
    "input": "How can I build a RAG application using LangChain?"
}

response = rag_pipeline.invoke(query)

print(response)

Below is a practical, step‑by‑step recipe for building a Retrieval‑Augmented Generation (RAG) app with **LangChain**.  
Everything is drawn from the concepts mentioned in the context, so you can see why each piece matters and how to keep the pipeline as simple (or as sophisticated) as you need.

---

## 1. Gather the source material  

| Source type | LangChain loader | What it does |
|-------------|------------------|--------------|
| Plain‑text files | `TextLoader` | Reads a `.txt` (or any simple text) file into a `Document`. |
| PDFs | `PyPDFLoader`, `PDFMinerLoader`, etc. | Extracts the textual content from each page of a PDF. |
| Web pages | `WebBaseLoader`, `RecursiveUrlLoader` | Crawls a URL (or a site) and returns the page text. |
| …other formats (CSV, DOCX, Notion, etc.) | Corresponding loaders | Same pattern – turn external data into LangChain `Document`s. |

**Tip:** Load only the data you actually need; a smaller corpus = lower latency and cost.

```python
from langchain.d

In [ ]:
query = {
    "input": "What is the difference between dense and sparse retrieval?"
}

response = rag_pipeline.invoke(query)

print(response)

**Dense retrieval** and **sparse retrieval** are two complementary ways of finding documents:

| Aspect | Dense Retrieval | Sparse Retrieval |
|--------|----------------|------------------|
| **How it works** | Represents queries and documents as dense vector embeddings (e.g., from a neural encoder) and finds matches by measuring vector similarity (e.g., cosine similarity). | Represents text as high‑dimensional sparse vectors of term frequencies or TF‑IDF weights (lexical/keyword representation) and matches by exact or near‑exact term overlap (e.g., BM25). |
| **Strengths** | Captures semantic meaning, so it can retrieve relevant items even when the query and document use different words (e.g., “automobile maintenance” ↔ “car maintenance”). | Excels at precise lexical matching, making it ideal for technical names, product IDs, error messages, or any exact‑term queries. |
| **Weaknesses** | May miss documents that contain the exact keyword but are semantically distant; can be less relia

In [ ]:
query = {
    "input": "What is semantic chunking?"
}

response = rag_pipeline.invoke(query)

print(response)

**Semantic chunking** is a document‑splitting method that decides where to break a text based on its meaning rather than on fixed sizes or simple character counts. It analyzes the semantic relationships between sentences (or other textual units) and creates chunks that keep related sentences together, ensuring that each chunk represents a coherent piece of information. This contrasts with purely syntactic chunking methods (e.g., recursive character splitting) that ignore the underlying content.
